# ARC-AGI-3 Exact Competition Scoring — Strict No Task-Specific Prior

This notebook is a clean rebuild of the supplied Duck Harness Kaggle notebook.

## Operational definition

“Strict no task-specific prior” means the agent receives no:

- public-game routebooks or hard-coded action sequences;
- public/hidden game-ID branches;
- teacher trajectories, distilled target actions, demonstrations, or replay buffers;
- state fingerprints mapped to known actions;
- cross-game win banking or transfer;
- multi-pass best-of selection.

The general pretrained Qwen model remains a **general architectural/model prior**. Therefore this notebook is not literally prior-free. It is designed to be free of ARC-AGI-3 game-specific solution priors.

## Scoring guarantee

The notebook uses the official `OperationMode.COMPETITION` gateway during a Kaggle competition rerun. The official scorer—not this notebook—computes the hidden score. A local/offline run is only a public-environment diagnostic and cannot reproduce or predict the hidden leaderboard result.


In [ ]:
import hashlib
import inspect
import json
import math
import os
import pickle
import re
import subprocess
import sys
import sysconfig
import time
import types
from collections.abc import Mapping, Sequence, Set
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {
    "1",
    "true",
}
NOTEBOOK_START_EPOCH = time.time()

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

os.environ.update(
    {
        "TAAF_DISABLE_ROUTEBOOK": "1",
        "TAAF_DISABLE_REPLAY": "1",
        "TAAF_DISABLE_TEACHER_TRACES": "1",
        "TAAF_DISABLE_DISTILLATION": "1",
        "TAAF_DISABLE_CROSS_GAME_TRANSFER": "1",
        "TAAF_DISABLE_WIN_BANKING": "1",
        "TAAF_FRESH_GAME_SESSION": "1",
    }
)

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [
        cuda_library_path,
        *os.environ.get("LIBRARY_PATH", "").split(os.pathsep),
    ]
    if entry
)

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
SUBMISSION_PATH.unlink(missing_ok=True)

print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
print("taaf.kaggle: strict no ARC-game-specific prior mode")


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal engine-compatibility correction

The bundled harness already exposes `ACTION7` to the model when the environment makes it valid, but one reverse mapping may be absent. This cell only restores the label-to-engine round trip. It does not add instructions, animation metadata, game semantics, routes, or extra observations.


In [ ]:
import inference.agent.action_names as action_names

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"

assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

ACTION_COMPATIBILITY_STATUS = {
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "extra_observation_metadata": False,
}
print(f"taaf.kaggle: action compatibility = {ACTION_COMPATIBILITY_STATUS}")


## 5. Load the unmodified benchmark and deployment target

The pickles are restored exactly as supplied by the source bundle. The following cell then audits the live solver and rejects non-empty task-specific prior structures.


In [ ]:
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

if getattr(bm, "game_runs", None):
    raise RuntimeError(
        "benchmark_initial.pkl already contains completed game runs; refusing a non-pristine benchmark."
    )


## 6. Strict no-task-prior configuration and audit

This score-control plane uses the stock solver only—no graft package, teacher, replay, routebook, public-ID policy, or same-run cross-game transfer.

The audit is fail-closed: if the restored solver contains a non-empty object whose field name indicates task-specific solutions, or if its state is keyed by a preloaded environment identifier, execution stops before the benchmark begins.


In [ ]:
SCORE_MODE = "strict_no_task_prior"
LOCAL_FAST_EVAL_SECONDS = float(os.environ.get("TAAF_LOCAL_FAST_EVAL_SECONDS", "1500"))
TARGET_CONCURRENCY = max(1, int(os.environ.get("TAAF_CONCURRENCY", "4")))

_original_game_budget = float(getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0)
_original_concurrency = max(1, int(getattr(bm.solver, "concurrency", 1) or 1))

if not TRUE_SUBMISSION and LOCAL_FAST_EVAL_SECONDS > 0:
    bm.solver.max_runtime_s_per_game = (
        min(_original_game_budget, LOCAL_FAST_EVAL_SECONDS)
        if _original_game_budget > 0
        else LOCAL_FAST_EVAL_SECONDS
    )
elif TRUE_SUBMISSION:
    bm.solver.max_runtime_s_per_game = _original_game_budget

bm.solver.concurrency = min(_original_concurrency, TARGET_CONCURRENCY)
bm.n_passes = 1
bm.game_weights = None

_PRIOR_FIELD_RE = re.compile(
    r"(?:^|_)("
    r"routebook|routes?|solution(?:s|_map)?|answer_key|"
    r"teacher(?:_trace|_output|_actions?)?|distill(?:ed|ation)?|"
    r"demonstrations?|exemplars?|replay(?:_buffer|_bank)?|"
    r"trajector(?:y|ies)(?:_bank)?|win_bank|banked_wins?|"
    r"fingerprint(?:_map|_policy)?|game_lookup|game_policy|"
    r"public_game(?:s|_ids?)?|known_game(?:s|_ids?)?|"
    r"memorized(?:_actions?|_policy)?"
    r")(?:_|$)",
    flags=re.IGNORECASE,
)

_ATOMIC_TYPES = (str, bytes, bytearray, int, float, bool, type(None), Path)


def _game_identifier(game):
    for attr in ("env_name", "game_id", "name"):
        value = getattr(game, attr, None)
        if value:
            return str(value)
    return None


def _is_nonempty(value):
    if value is None:
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (str, bytes, bytearray, Mapping, Sequence, Set)):
        return len(value) > 0
    return True


def _iter_object_graph(root, max_depth=4, max_items_per_container=256):
    # Avoid descending into tensors, modules, callables, or large array objects.
    seen = set()
    stack = [("solver", root, 0)]

    while stack:
        path, value, depth = stack.pop()
        object_id = id(value)
        if object_id in seen:
            continue
        seen.add(object_id)
        yield path, value

        if depth >= max_depth or isinstance(value, _ATOMIC_TYPES):
            continue
        if isinstance(value, (types.ModuleType, type)) or callable(value):
            continue
        module_name = type(value).__module__
        if module_name.startswith(("numpy", "torch", "pandas", "PIL")):
            continue

        children = []
        if isinstance(value, Mapping):
            for index, (key, child) in enumerate(value.items()):
                if index >= max_items_per_container:
                    break
                children.append((f"{path}[{key!r}]", child))
        elif isinstance(value, (list, tuple, set, frozenset)):
            for index, child in enumerate(value):
                if index >= max_items_per_container:
                    break
                children.append((f"{path}[{index}]", child))
        elif hasattr(value, "__dict__"):
            for index, (name, child) in enumerate(vars(value).items()):
                if index >= max_items_per_container:
                    break
                children.append((f"{path}.{name}", child))

        for child_path, child in reversed(children):
            stack.append((child_path, child, depth + 1))


_preloaded_game_ids = {
    identifier
    for identifier in (_game_identifier(game) for game in getattr(bm, "games", []))
    if identifier
}
_preloaded_game_bases = {identifier.split("-", 1)[0] for identifier in _preloaded_game_ids}
_forbidden_identifiers = _preloaded_game_ids | _preloaded_game_bases

_prior_field_hits = []
_game_key_hits = []

for object_path, value in _iter_object_graph(bm.solver):
    field_name = object_path.rsplit(".", 1)[-1].split("[", 1)[0]
    if _PRIOR_FIELD_RE.search(field_name) and _is_nonempty(value):
        _prior_field_hits.append(
            {
                "path": object_path,
                "type": type(value).__name__,
                "preview": repr(value)[:240],
            }
        )

    if isinstance(value, Mapping):
        for key in list(value.keys())[:256]:
            if isinstance(key, str) and key in _forbidden_identifiers:
                _game_key_hits.append({"path": object_path, "key": key})
    elif isinstance(value, str) and value in _forbidden_identifiers:
        _game_key_hits.append({"path": object_path, "value": value})

if _prior_field_hits or _game_key_hits:
    raise RuntimeError(
        "Task-specific prior audit failed:\n"
        + json.dumps(
            {
                "prior_fields": _prior_field_hits,
                "game_keyed_state": _game_key_hits,
            },
            indent=2,
            sort_keys=True,
        )
    )

try:
    _solver_source = inspect.getsource(type(bm.solver))
except (OSError, TypeError):
    _solver_source = ""

NO_TASK_PRIOR_MANIFEST = {
    "mode": SCORE_MODE,
    "definition": "no ARC-AGI-3 game-specific solution prior",
    "general_pretrained_model_allowed": True,
    "teacher_supervision": False,
    "distilled_teacher_actions": False,
    "routebooks": False,
    "replays": False,
    "state_fingerprint_action_maps": False,
    "cross_game_transfer": False,
    "same_game_context_memory": True,
    "passes": int(bm.n_passes),
    "game_weights": bm.game_weights,
    "concurrency": int(bm.solver.concurrency),
    "serialized_per_game_budget_s": _original_game_budget,
    "active_per_game_budget_s": float(
        getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
    ),
    "solver_class": f"{type(bm.solver).__module__}.{type(bm.solver).__qualname__}",
    "solver_source_sha256": hashlib.sha256(_solver_source.encode("utf-8")).hexdigest()
    if _solver_source
    else None,
    "pre_run_prior_field_hits": _prior_field_hits,
    "pre_run_game_key_hits": _game_key_hits,
}

(WORKING_DIR / "no_task_prior_manifest.json").write_text(
    json.dumps(NO_TASK_PRIOR_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(json.dumps(NO_TASK_PRIOR_MANIFEST, indent=2, sort_keys=True))


## 7. Run through the official scoring path

During a competition rerun, the notebook connects to the Kaggle ARC gateway, enumerates every environment, opens each once in `OperationMode.COMPETITION`, executes one pass, and validates the resulting `submission.parquet`.

Competition mode scores every available environment, permits only level resets, prevents repeated `make()` calls for an environment, and does not reveal the in-flight scorecard.


In [ ]:
def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    if len(game_ids) != len(set(game_ids)):
        raise RuntimeError("Competition Arcade exposed duplicate environment IDs.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    if len(game_ids) != len(set(game_ids)):
        raise RuntimeError("Offline Arcade exposed duplicate environment IDs.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


def _validate_submission(path: Path) -> dict:
    import pandas as pd

    if not path.is_file():
        raise RuntimeError(f"Expected submission output was not created: {path}")

    frame = pd.read_parquet(path)
    required_columns = {"row_id", "game_id", "end_of_game", "score"}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise RuntimeError(f"submission.parquet is missing columns: {missing}")
    if frame.empty:
        raise RuntimeError("submission.parquet contains zero rows.")
    if frame["row_id"].isna().any() or frame["row_id"].astype(str).duplicated().any():
        raise RuntimeError("submission.parquet has null or duplicate row_id values.")
    if frame["game_id"].isna().any():
        raise RuntimeError("submission.parquet has null game_id values.")

    scores = pd.to_numeric(frame["score"], errors="coerce")
    if scores.isna().any() or not scores.map(math.isfinite).all():
        raise RuntimeError("submission.parquet contains non-numeric or non-finite scores.")
    if (scores < 0).any():
        raise RuntimeError("submission.parquet contains negative scores.")

    summary = {
        "path": str(path),
        "rows": int(len(frame)),
        "columns": [str(column) for column in frame.columns],
        "games": int(frame["game_id"].astype(str).nunique()),
        "end_of_game_rows": int(frame["end_of_game"].astype(bool).sum()),
        "score_min": float(scores.min()),
        "score_max": float(scores.max()),
        "score_mean_in_file": float(scores.mean()),
    }
    (WORKING_DIR / "submission_validation.json").write_text(
        json.dumps(summary, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return summary


print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    competition_env_files = str(
        Path(
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
        ).parent
        / "environment_files"
    )
    bm.games = _offline_games(competition_env_files)

assert bm.n_passes == 1
assert bm.game_weights is None
assert os.environ["TAAF_DISABLE_CROSS_GAME_TRANSFER"] == "1"
assert os.environ["TAAF_DISABLE_WIN_BANKING"] == "1"

soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
            seconds=budget - min(600.0, budget / 2)
        )

SUBMISSION_PATH.unlink(missing_ok=True)

try:
    await bm.run(
        soft_end_time=soft_end,
        runtime_environment=target,
        minimal_diagnostics=TRUE_SUBMISSION,
    )

    if TRUE_SUBMISSION:
        SUBMISSION_VALIDATION = _validate_submission(SUBMISSION_PATH)
        print(
            "OFFICIAL COMPETITION OUTPUT VALIDATED:\n"
            + json.dumps(SUBMISSION_VALIDATION, indent=2, sort_keys=True)
        )
    else:
        import pandas as pd

        pd.DataFrame(
            [["local_sentinel_0", "local_sentinel", True, 0.0]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(SUBMISSION_PATH, index=False)
        print(
            "LOCAL RUN COMPLETE: wrote a zero-valued submission sentinel. "
            "The competition rerun will delete and replace it."
        )
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")

## 9. Local score card and provenance

The card below reports the benchmark’s own local summary. It is not transformed into a hidden-score estimate. The no-task-prior manifest is displayed so the executed configuration is auditable.


In [ ]:
import re
from html import escape
from IPython.display import HTML, display

manifest_path = WORKING_DIR / "no_task_prior_manifest.json"
if manifest_path.is_file():
    executed_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("NO-TASK-PRIOR MANIFEST")
    print(json.dumps(executed_manifest, indent=2, sort_keys=True))

if TRUE_SUBMISSION:
    print(
        "Official competition rerun completed. "
        "The hidden score is intentionally unavailable inside the run."
    )
elif not getattr(bm, "game_runs", None):
    print("No completed local benchmark data is available.")
else:
    from taaf import diagnostics as taaf_diagnostics

    summary_text = taaf_diagnostics.run_summary_text(bm)

    def _extract(pattern, cast=str, default=None):
        match = re.search(pattern, summary_text, flags=re.MULTILINE)
        if not match:
            return default
        try:
            return cast(match.group(1).strip())
        except Exception:
            return default

    mean_score = _extract(r"^mean score:\s*([0-9.]+)\s*$", float, 0.0)
    median_score = _extract(r"^median score:\s*([0-9.]+)\s*$", float, 0.0)
    duration = _extract(r"^duration:\s*(.+?)\s*$", str, "unknown")
    games = _extract(r"^games:\s*(\d+)\s*$", int, 0)
    won = _extract(r"^runs:\s*\d+\s*\(won:\s*(\d+)\)\s*$", int, 0)
    actions = _extract(r"^total actions:\s*(\d+)\s*$", int, 0)
    tokens = _extract(r"^total tokens:\s*(\d+)\s*$", int, 0)

    per_game = re.findall(
        r"^\s+\S+:\s+score=([0-9.]+),\s+levels=([0-9.]+)/([0-9.]+),",
        summary_text,
        flags=re.MULTILINE,
    )
    positive = sum(float(score) > 0 for score, _, _ in per_game)
    levels_done = sum(float(done) for _, done, _ in per_game)
    levels_total = sum(float(total) for _, _, total in per_game)

    display(
        HTML(
            f"""
            <div style="border:1px solid #6b7280;border-radius:14px;padding:20px 24px;margin:14px 0;max-width:920px;font-family:Arial,sans-serif">
              <div style="font-size:15px;font-weight:800;letter-spacing:.05em">ARC-AGI-3 STRICT NO TASK-SPECIFIC PRIOR — LOCAL RUN</div>
              <div style="font-size:46px;font-weight:850;line-height:1.15;margin-top:8px">{mean_score:.2f}<span style="font-size:18px;font-weight:500"> / 100</span></div>
              <div style="font-size:14px;margin-top:4px">Public/offline diagnostic from the benchmark scorer; not a hidden leaderboard estimate.</div>
              <hr style="margin:16px 0;border:none;border-top:1px solid #6b7280">
              <table style="border-collapse:collapse;width:100%;font-size:14px;line-height:1.9">
                <tr><td>Median score</td><td><b>{median_score:.2f}</b></td><td>Games fully solved</td><td><b>{won}/{games}</b></td></tr>
                <tr><td>Games with positive score</td><td><b>{positive}/{len(per_game)}</b></td><td>Levels completed</td><td><b>{levels_done:.0f}/{levels_total:.0f}</b></td></tr>
                <tr><td>Total actions</td><td><b>{actions:,}</b></td><td>Total generated tokens</td><td><b>{tokens:,}</b></td></tr>
                <tr><td>Benchmark duration</td><td><b>{escape(duration)}</b></td><td>Cross-game transfer</td><td><b>disabled</b></td></tr>
              </table>
            </div>
            """
        )
    )
